# VeReMi Paper 1 — Experimental Protocol and Leakage-Free Archive Split

Input: `GRAPH_DATASET_ROOT / "archives"`

Output: `INITIAL_SPLIT_ROOT`

The **archive** is the split unit. Target partition: **157 train / 34 validation / 34 test**. The notebook first discovers the actual label/scenario structure rather than assuming it.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, re

from src.config import GRAPH_DATASET_ROOT, INITIAL_SPLIT_ROOT
ROOT = GRAPH_DATASET_ROOT / "archives"
OUT = INITIAL_SPLIT_ROOT
OUT.mkdir(parents=True, exist_ok=True)

SEED = 20260811
TRAIN_N, VAL_N, TEST_N, EXPECTED_N = 157, 34, 34, 225

archive_dirs = sorted(p for p in ROOT.iterdir() if p.is_dir())
print("Archives found:", len(archive_dirs))
assert len(archive_dirs) == EXPECTED_N


In [ ]:
# 1. Inspect schema
sample = pd.read_parquet(archive_dirs[0] / "edge_features.parquet")
print("Sample archive:", archive_dirs[0].name)
print("Rows:", len(sample))
print("\nColumns:")
for c in sample.columns:
    print(c)

candidate_label_cols = [
    c for c in sample.columns
    if any(k in c.lower() for k in ["attack", "label", "class", "malicious"])
]
print("\nCandidate label columns:", candidate_label_cols)


In [ ]:
# 2. Archive inventory
records=[]
for i,d in enumerate(archive_dirs,1):
    if i==1 or i%25==0: print(f"Inspecting {i}/{len(archive_dirs)}")
    edge=pd.read_parquet(d/"edge_features.parquet")
    rec={"archive_name":d.name,
         "edge_rows":len(edge),
         "snapshot_count":edge["snapshot_id"].nunique() if "snapshot_id" in edge.columns else np.nan}
    for col in candidate_label_cols:
        vals=edge[col].dropna()
        rec[f"{col}__unique"]=vals.nunique()
        rec[f"{col}__values"]="|".join(map(str,sorted(vals.astype(str).unique())))
    m=re.search(r"uc(\d+)\.(\d+)\.",d.name)
    rec["use_case"]=m.group(1) if m else None
    rec["run_id"]=m.group(2) if m else None
    records.append(rec)

inventory=pd.DataFrame(records)
inventory.to_csv(OUT/"01_archive_inventory.csv",index=False)
display(inventory.head())


In [ ]:
# 3. Candidate-label audit
summary=[]
for col in candidate_label_cols:
    u=f"{col}__unique"; v=f"{col}__values"
    summary.append({
        "column":col,
        "archives_with_value":int(inventory[u].notna().sum()),
        "archives_single_value":int((inventory[u]==1).sum()),
        "archives_multi_value":int((inventory[u]>1).sum()),
        "distinct_archive_signatures":int(inventory[v].nunique(dropna=True))
    })
label_summary_df=pd.DataFrame(summary)
label_summary_df.to_csv(OUT/"02_candidate_label_summary.csv",index=False)
display(label_summary_df)

preferred=None
for exact in ["attackerType","attacker_type","attack_type"]:
    if exact in candidate_label_cols:
        preferred=exact; break
if preferred is None and candidate_label_cols:
    preferred=candidate_label_cols[0]

if preferred:
    inventory["stratum"]=inventory[f"{preferred}__values"].fillna("NO_LABEL_SIGNATURE")
    print("Selected stratification source:",preferred)
else:
    inventory["stratum"]="ALL_ARCHIVES"
    print("WARNING: no label-like column found; using archive-only random split.")

stratum_counts=inventory["stratum"].value_counts(dropna=False).rename_axis("stratum").reset_index(name="archives")
display(stratum_counts)


In [ ]:
# 4. Exact proportional allocation
rng=np.random.default_rng(SEED)
work=inventory.copy()
work["_rand"]=rng.random(len(work))
work=work.sort_values(["stratum","_rand"]).reset_index(drop=True)

strata=work.groupby("stratum",dropna=False).size().rename("n").reset_index()
strata["train_raw"]=strata["n"]*TRAIN_N/EXPECTED_N
strata["train_n"]=np.floor(strata["train_raw"]).astype(int)
remaining=TRAIN_N-strata["train_n"].sum()
for idx in (strata["train_raw"]-strata["train_n"]).sort_values(ascending=False).index[:remaining]:
    strata.loc[idx,"train_n"]+=1

strata["val_raw"]=strata["n"]*VAL_N/EXPECTED_N
strata["val_n"]=np.floor(strata["val_raw"]).astype(int)
strata["capacity"]=strata["n"]-strata["train_n"]
strata["val_n"]=np.minimum(strata["val_n"],strata["capacity"])
remaining=VAL_N-strata["val_n"].sum()
for idx in (strata["val_raw"]-np.floor(strata["val_raw"])).sort_values(ascending=False).index:
    if remaining==0: break
    if strata.loc[idx,"val_n"]<strata.loc[idx,"capacity"]:
        strata.loc[idx,"val_n"]+=1; remaining-=1
if remaining:
    for idx in strata.index:
        while remaining and strata.loc[idx,"val_n"]<strata.loc[idx,"capacity"]:
            strata.loc[idx,"val_n"]+=1; remaining-=1

strata["test_n"]=strata["n"]-strata["train_n"]-strata["val_n"]
assert strata.train_n.sum()==TRAIN_N and strata.val_n.sum()==VAL_N and strata.test_n.sum()==TEST_N
display(strata[["stratum","n","train_n","val_n","test_n"]])


In [ ]:
# 5. Assign archives
quota=strata.set_index("stratum")[["train_n","val_n","test_n"]].to_dict("index")
parts=[]
for s,g in work.groupby("stratum",dropna=False,sort=False):
    g=g.copy().reset_index(drop=True); q=quota[s]
    a=q["train_n"]; b=q["val_n"]
    g["split"]=["train"]*a+["validation"]*b+["test"]*(len(g)-a-b)
    parts.append(g)

manifest=pd.concat(parts,ignore_index=True).drop(columns="_rand")
manifest=manifest.sort_values(["split","archive_name"]).reset_index(drop=True)
manifest.to_csv(OUT/"03_full_split_manifest.csv",index=False)

for s,f in [("train","train_archives.csv"),("validation","validation_archives.csv"),("test","test_archives.csv")]:
    manifest[manifest.split==s].to_csv(OUT/f,index=False)

display(manifest["split"].value_counts().to_frame("archives"))


In [ ]:
# 6. Isolation audit
train=set(manifest.loc[manifest.split=="train","archive_name"])
val=set(manifest.loc[manifest.split=="validation","archive_name"])
test=set(manifest.loc[manifest.split=="test","archive_name"])
assert not(train&val) and not(train&test) and not(val&test)
print("Train/validation overlap:",len(train&val))
print("Train/test overlap:",len(train&test))
print("Validation/test overlap:",len(val&test))


In [ ]:
# 7. Edge-level target distribution (audit only)
rows=[]
if preferred:
    for split in ["train","validation","test"]:
        names=set(manifest.loc[manifest.split==split,"archive_name"])
        counts={}
        for d in archive_dirs:
            if d.name not in names: continue
            x=pd.read_parquet(d/"edge_features.parquet",columns=[preferred])
            for label,n in x[preferred].value_counts(dropna=False).items():
                counts[str(label)]=counts.get(str(label),0)+int(n)
        total=sum(counts.values())
        for label,n in counts.items():
            rows.append({"split":split,"target_column":preferred,"label":label,
                         "edge_count":n,"fraction":n/total if total else np.nan})

target_distribution=pd.DataFrame(rows)
target_distribution.to_csv(OUT/"04_target_distribution_by_split.csv",index=False)
display(target_distribution)


In [ ]:
# 8. Freeze protocol and final validation
protocol={
 "dataset":"VeReMi Graph Dataset V1.0",
 "paper":"Paper 1 - Graph Construction Methodology",
 "random_seed":SEED,
 "split_unit":"archive",
 "train_archives":TRAIN_N,"validation_archives":VAL_N,"test_archives":TEST_N,
 "selected_target_column_for_distribution_audit":preferred,
 "leakage_controls":[
   "No archive appears in more than one split.",
   "No random node, edge, or snapshot split across archive partitions.",
   "All learned preprocessing must be fitted on training archives only.",
   "Validation and test archives remain unseen during model fitting.",
   "Identifiers and ground-truth/label-derived variables are excluded from model inputs."
 ]}
with open(OUT/"05_experimental_protocol.json","w",encoding="utf-8") as f:
    json.dump(protocol,f,indent=2)

passed=(len(manifest)==225 and len(train)==157 and len(val)==34 and len(test)==34
        and manifest.archive_name.nunique()==225 and not(train&val) and not(train&test) and not(val&test))

final=pd.DataFrame({
 "check":["Total archives","Train archives","Validation archives","Test archives",
          "Unique archives","Train-validation overlap","Train-test overlap",
          "Validation-test overlap","Fixed random seed","FINAL_SPLIT_STATUS"],
 "value":[len(manifest),len(train),len(val),len(test),manifest.archive_name.nunique(),
          len(train&val),len(train&test),len(val&test),SEED,"PASS" if passed else "REVIEW_REQUIRED"]
})
final.to_csv(OUT/"06_FINAL_SPLIT_VALIDATION.csv",index=False)
display(final)
print("\nFinal split status:","PASS" if passed else "REVIEW_REQUIRED")
print("Outputs:",OUT)


## Stop point

Before constructing PyTorch Geometric objects, send back the displayed:
1. candidate label summary,
2. archive strata/allocation,
3. target distribution by split,
4. final validation.

We will confirm the prediction target and balance before tensor construction.
